# Lifetimes

Welcome! Lifetimes are one of Rust's most distinctive features. They help the compiler ensure that references never outlive the data they point to — preventing *dangling references*, a common source of bugs in other languages. Think of lifetimes as "how long does this reference stay valid?" This guide walks you through what they are, why they matter, and how to use them confidently.

## What Are Lifetimes and Why Do They Matter?

A **lifetime** is how long a reference remains valid. When you borrow data (with `&` or `&mut`), that reference must not outlive the original data. If it did, you'd have a **dangling reference** — a pointer to memory that's been freed. In C or C++, this causes undefined behavior. Rust's borrow checker uses lifetimes to prevent this at compile time.

**Analogy:** Imagine lending a book to a friend. The "lifetime" of that loan is until your friend returns it. You can't sell the book while it's still loaned out — and Rust enforces similar rules for references.

## The Problem: Dangling References

Without lifetimes, this would be possible (and dangerous):

```rust
// This would NOT compile in Rust!
fn dangling() -> &str {
    let s = String::from("hello");
    &s  // s is dropped here; returning &s would be a dangling reference!
}
```

Rust rejects this because `s` is dropped at the end of the function, so returning `&s` would point to freed memory. Lifetimes make this relationship explicit.

## Basic Lifetime Annotations: `'a`

Lifetime parameters use apostrophes and lowercase names, like `'a`, `'b`, or `'static`. They describe *how long* references live relative to each other, not *how long* in absolute time.

A function that takes two references and returns one needs to tell the compiler: "the returned reference will be valid as long as the shorter of the two input lifetimes."

In [ ]:
// Without explicit lifetimes, the compiler infers them.
// Here we add them explicitly to show the relationship.
fn longest<'a>(x: &'a str, y: &'a str) -> &'a str {
    if x.len() > y.len() { x } else { y }
}

let s1 = "short";
let s2 = "longer string";
let result = longest(s1, s2);
println!("Longer: {}", result);

## Lifetime Elision: The Three Rules

Often you don't need to write lifetimes at all! The compiler applies **lifetime elision rules** to infer them:

1. **Rule 1:** Each reference parameter gets its own lifetime. `fn foo(x: &str, y: &str)` becomes `fn foo<'a, 'b>(x: &'a str, y: &'b str)`.

2. **Rule 2:** If there's exactly one input lifetime, it's assigned to all output lifetimes. `fn foo(x: &str) -> &str` becomes `fn foo<'a>(x: &'a str) -> &'a str`.

3. **Rule 3:** If there are multiple input lifetimes but one is `&self` or `&mut self`, that lifetime is assigned to all output lifetimes.

If the compiler can't figure it out after these rules, it will ask you to add explicit annotations.

In [ ]:
// Elision in action: no explicit lifetimes needed!
fn first_word(s: &str) -> &str {
    s.split_whitespace().next().unwrap_or("")
}

let text = "hello world";
let word = first_word(text);
println!("First word: {}", word);

## Lifetimes in Function Signatures

When you have multiple references and a return type that's a reference, you must tie them together. The returned reference's lifetime must not exceed any of the input lifetimes.

In [ ]:
fn choose_longer<'a>(a: &'a str, b: &'a str) -> &'a str {
    if a.len() >= b.len() { a } else { b }
}

let a = "apple";
let b = "banana";
let longer = choose_longer(a, b);
println!("Longer: {}", longer);

## Lifetimes in Structs

When a struct holds a reference, it must specify how long that reference lives. The struct cannot outlive the data it borrows.

In [ ]:
struct Excerpt<'a> {
    text: &'a str,
}

let novel = String::from("Call me Ishmael. Some years ago...");
let first_sentence = novel.split('.').next().unwrap_or("");
let excerpt = Excerpt { text: first_sentence };
println!("Excerpt: {}", excerpt.text);

## The `'static` Lifetime

The special lifetime `'static` means "lives for the entire program." String literals have type `&'static str` because they're baked into the binary. Use `'static` when you need a reference that never goes away.

In [ ]:
// String literals are &'static str
let greeting: &'static str = "Hello, world!";

// Sometimes you need to constrain a generic to 'static
fn print_static<T: std::fmt::Display + 'static>(x: T) {
    println!("{}", x);
}

print_static(greeting);
print_static(42);  // i32 is 'static (no references)

## Lifetime Bounds on Generics

When combining lifetimes with generics, you can add lifetime bounds. For example, `T: 'a` means "all references in T must outlive `'a`."

In [ ]:
use std::fmt::Display;

fn longest_with_ann<'a, T>(x: &'a str, y: &'a str, ann: T) -> &'a str
where
    T: Display,
{
    println!("Announcement: {}", ann);
    if x.len() > y.len() { x } else { y }
}

let s1 = "hi";
let s2 = "hello";
let result = longest_with_ann(s1, s2, "Comparing strings");
println!("Result: {}", result);

## Common Lifetime Patterns

**Pattern 1:** Returning a slice from a function — the output lifetime is tied to the input.

**Pattern 2:** Structs that borrow — the struct's lifetime parameter ties it to the borrowed data.

**Pattern 3:** Methods on structs with references — `&self` or `&mut self` often determines the output lifetime.

In [ ]:
// Pattern: string processing that returns slices
fn get_until_space(s: &str) -> &str {
    match s.find(' ') {
        Some(i) => &s[..i],
        None => s,
    }
}

let input = "hello world";
let first = get_until_space(input);
println!("First word: {}", first);

## Why the Borrow Checker Needs Lifetimes

The borrow checker tracks:
- **Who** owns data
- **Who** borrows it (and whether mutably)
- **How long** each borrow lasts

Lifetimes answer "how long." Without them, the compiler couldn't verify that your references are always valid. They're the glue that makes Rust's memory safety guarantees work.

## Lifetime Subtyping

Lifetime subtyping: if `'a` outlives `'b` (written `'a: 'b`), then `'a` is a subtype of `'b`. A reference with lifetime `'a` can be used where `'b` is expected, because it lives at least as long.

This is useful when a struct holds a reference with a shorter lifetime than the struct's own lifetime parameter.

In [ ]:
struct Wrapper<'a> {
    inner: &'a str,
}

impl<'a> Wrapper<'a> {
    fn get(&self) -> &'a str {
        self.inner
    }
}

let s = String::from("wrapped");
let w = Wrapper { inner: &s };
println!("Got: {}", w.get());

## Practical Example: Multiple String Slices

Here's a more realistic example: a function that finds a substring and returns a slice into the original string.

In [ ]:
fn find_substring<'a>(haystack: &'a str, needle: &str) -> Option<&'a str> {
    haystack.find(needle).map(|i| &haystack[i..i + needle.len()])
}

let text = "The quick brown fox jumps over the lazy dog";
let sub = find_substring(text, "brown");
println!("Found: {:?}", sub);

## Common Mistakes Beginners Make

1. **Over-annotating** — Try without explicit lifetimes first; elision often works.
2. **Using the same lifetime for everything** — Sometimes you need `'a` and `'b` to express different relationships.
3. **Confusing lifetime with scope** — A variable's scope is where it's valid; lifetime is about references.
4. **Returning references to local data** — Always invalid; the compiler will catch it.
5. **Forgetting `'static` for string literals** — They're special; they live forever.

## Key Rules to Remember

- Lifetimes prevent dangling references by tying reference validity to the borrowed data.
- Use `'a`, `'b`, etc. to name lifetimes; `'static` means "lives for the whole program."
- Lifetime elision has three rules; add explicit annotations when the compiler asks.
- Structs holding references need a lifetime parameter.
- The returned reference's lifetime cannot exceed any input reference's lifetime.
- When in doubt, let the compiler guide you — it will suggest fixes!